This script will fetch and format raw data from space and earth weather from Jan 1st, 2021 - jan 1st 2026.

1. Fetch space data by downloading a file from NASA OMNIWeb Data, source: https://omniweb.gsfc.nasa.gov/form/dx1.html

In [13]:
import pandas as pd
import numpy as np
import requests

In [14]:
### read the raw data file
df_space = pd.read_csv("../../data/raw/omni2_Qj2SZdRRin.lst.txt", sep=r"\s+", engine="python", names=["year", "day", "hour", "Bz", "solar_wind_speed", "Kp_x10"])
df_space.head()

,year,day,hour,Bz,solar_wind_speed,Kp_x10
0,2021,1,0,1.6,365.0,0
1,2021,1,1,1.5,366.0,0
2,2021,1,2,-0.3,360.0,0
3,2021,1,3,1.8,358.0,3
4,2021,1,4,1.5,356.0,3


In [15]:

## Convert the date and time columns into a single datetime column
df_space['datetime'] = pd.to_datetime(df_space['year'] * 1000 + df_space['day'], format='%Y%j') + pd.to_timedelta(df_space['hour'], unit='h')

## transform Kp_x10 into Kp index by dividing by 10
df_space['Kp'] = df_space['Kp_x10'] / 10

### drop some columns
df_space = df_space.drop(['year', 'day', 'hour', 'Kp_x10'], axis=1)
df_space.head()

,Bz,solar_wind_speed,datetime,Kp
0,1.6,365.0,2021-01-01 00:00:00,0.0
1,1.5,366.0,2021-01-01 01:00:00,0.0
2,-0.3,360.0,2021-01-01 02:00:00,0.0
3,1.8,358.0,2021-01-01 03:00:00,0.3
4,1.5,356.0,2021-01-01 04:00:00,0.3


In [16]:
df_space.info()

<class 'pandas.DataFrame'>
RangeIndex: 43848 entries, 0 to 43847
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Bz                43848 non-null  float64       
 1   solar_wind_speed  43848 non-null  float64       
 2   datetime          43848 non-null  datetime64[us]
 3   Kp                43848 non-null  float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 1.3 MB


In [17]:
df_space.describe()

,Bz,solar_wind_speed,datetime,Kp
count,43848.000000,43848.000000,43848,43848.000000
mean,7.784779,586.790572,2023-07-03 11:30:00,1.967768
min,-42.200000,256.000000,2021-01-01 00:00:00,0.000000
25%,-1.800000,363.000000,2022-04-02 17:45:00,1.000000
50%,-0.100000,415.500000,2023-07-03 11:30:00,1.700000
75%,1.700000,491.000000,2024-10-02 05:15:00,2.700000
max,999.900000,9999.000000,2026-01-01 23:00:00,9.000000
std,88.417742,1206.106074,NaN,1.326060


In [18]:
df_space['Kp'].value_counts()
print(" the fraction of data points with Kp >= 5 is: ", (len(df_space[df_space['Kp'] >= 5]) / len(df_space))*100, "%")

 the fraction of data points with Kp >= 5 is:  2.832512315270936 %


The missing values in the raw data were defaulted to numbers with multiple 9's.
we will drop those rows

In [19]:
mask = (df_space['Bz'] != 999.9) & (df_space['solar_wind_speed'] != 9999.0) & (df_space['Kp'] != 9.9)
df_space = df_space[mask]
## reorganize the columns
df_space = df_space[['datetime', 'Bz', 'solar_wind_speed', 'Kp']]
df_space.head()

,datetime,Bz,solar_wind_speed,Kp
0,2021-01-01 00:00:00,1.6,365.0,0.0
1,2021-01-01 01:00:00,1.5,366.0,0.0
2,2021-01-01 02:00:00,-0.3,360.0,0.0
3,2021-01-01 03:00:00,1.8,358.0,0.3
4,2021-01-01 04:00:00,1.5,356.0,0.3


2. Fetch cloud percentage cover data file for both `Regina` and `Yellowknife` locations downloaded from open-meteo, source: https://open-meteo.com/en/docs/historical-forecast-api?hourly=cloud_cover

In [20]:
# df_weather = pd.read_csv("open-meteo-50.45N104.63W578m.csv")
## define the list of weather files and their corresponding latitude and longitude
files_weather = [
    {"file_name": "../../data/raw/open-meteo-50.45N104.63W578m.csv", "latitude": 50.45, "longitude": -104.63, "city": "Regina"},
    {"file_name": "../../data/raw/open-meteo-62.46N114.32W192m.csv", "latitude": 62.46, "longitude": -114.32, "city": "Yellowknife"}
]
# read the weather data files and concatenate them into a single dataframe
df_weather_list = []
for file in files_weather:
    df_weather = pd.read_csv(file["file_name"], skiprows=3)
    df_weather["latitude"] = file["latitude"]
    df_weather["longitude"] = file["longitude"]
    df_weather["city"] = file["city"]
    ## rename the columns with (unit)
    df_weather = df_weather.rename(columns={"cloud_cover (%)": "cloud_cover"})
    ## overwrite 'time' column from this format (yyyy-mm-ddThh:mm) to propper datetime format
    df_weather["datetime"] = pd.to_datetime(df_weather["time"], format='%Y-%m-%dT%H:%M')
    df_weather = df_weather.drop(columns=["time"])
    ## reorder the columns
    df_weather = df_weather[["datetime", "latitude", "longitude", "city", "cloud_cover"]]
    df_weather_list.append(df_weather)
df_weather = pd.concat(df_weather_list, ignore_index=True)
df_weather.head()

,datetime,latitude,longitude,city,cloud_cover
0,2021-01-01 00:00:00,50.45,-104.63,Regina,4
1,2021-01-01 01:00:00,50.45,-104.63,Regina,4
2,2021-01-01 02:00:00,50.45,-104.63,Regina,10
3,2021-01-01 03:00:00,50.45,-104.63,Regina,4
4,2021-01-01 04:00:00,50.45,-104.63,Regina,5


Merge the space and weather features for the same `datetime`

In [21]:
## Merge the space and weather features for the same datetime
df_merged = pd.merge(df_space, df_weather, on="datetime", how="inner")
## reoprganize the columns
df_merged = df_merged[["datetime", "latitude", "longitude", "city",  "Bz", "solar_wind_speed", "Kp", "cloud_cover"]]
## save to final dataframe to csv file
df_merged.to_csv("../../data/processed/data_aurora.csv", index=False)
df_merged.head()

,datetime,latitude,longitude,city,Bz,solar_wind_speed,Kp,cloud_cover
0,2021-01-01 00:00:00,50.45,-104.63,Regina,1.6,365.0,0.0,4
1,2021-01-01 00:00:00,62.46,-114.32,Yellowknife,1.6,365.0,0.0,81
2,2021-01-01 01:00:00,50.45,-104.63,Regina,1.5,366.0,0.0,4
3,2021-01-01 01:00:00,62.46,-114.32,Yellowknife,1.5,366.0,0.0,99
4,2021-01-01 02:00:00,50.45,-104.63,Regina,-0.3,360.0,0.0,10


In [23]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 86150 entries, 0 to 86149
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   datetime          86150 non-null  datetime64[us]
 1   latitude          86150 non-null  float64       
 2   longitude         86150 non-null  float64       
 3   city              86150 non-null  str           
 4   Bz                86150 non-null  float64       
 5   solar_wind_speed  86150 non-null  float64       
 6   Kp                86150 non-null  float64       
 7   cloud_cover       86150 non-null  int64         
dtypes: datetime64[us](1), float64(5), int64(1), str(1)
memory usage: 6.0 MB
